# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset ("Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells") using the `mlcroissant` library, referencing all entities via their `@id` IDs, as per the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# View the dataset metadata
print(f"Dataset: {dataset.metadata.name}")
print(dataset.metadata.description)

# Display available fields in the main metadata object
print(f"Published: {dataset.metadata.date_published}")
print(f"Version: {dataset.metadata.version}")
print(f"Identifier: {dataset.metadata.identifier}")

## 2. Data Overview
Review the available record sets and their fields, and display their respective `@id`s.

We will print all available RecordSets, their fields (with `@id`s), and columns. These RecordSet and Field IDs will be used throughout the notebook.

In [ ]:
from pprint import pprint

# List all RecordSets
print("Available RecordSets (@id):")
recordsets = list(dataset.metadata.record_sets)
for rs in recordsets:
    print(f"  RecordSet name: {rs.name}")
    print(f"    @id: {rs.id}")
    print(f"    Description: {rs.description}")
    # List Fields and Columns
    if hasattr(rs, 'fields'):
        print("    Fields:")
        for f in rs.fields:
            print(f"      {f.name} (@id: {f.id}, type: {f.data_type})")
    if hasattr(rs, 'columns') and rs.columns:
        print("    Columns:")
        for col in rs.columns:
            print(f"      {col.name} (@id: {col.id}, type: {col.data_type})")
    print()

# If there are no record sets, print a warning
if not recordsets:
    print("WARNING: No record sets discovered in the schema. Please check the schema definition.")

## 3. Data Extraction
Load data from a specific RecordSet into a DataFrame for analysis. We'll use the RecordSet and field `@id`s provided above.

Below, we first collect all RecordSet `@id`s, then extract their rows into pandas DataFrames. For demonstration, we will use the *first* RecordSet found (if multiple are present).

In [ ]:
# Collect all RecordSet @ids (URIs) from dataset metadata
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

# For demonstration, just use the first record set (if any exist)
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
    print(f"Loading records from RecordSet: {main_record_set_id}")
    records = list(dataset.records(record_set=main_record_set_id))
    df = pd.DataFrame(records)
    dataframes[main_record_set_id] = df

    print("Columns (fields) available:")
    print(df.columns.tolist())
    display(df.head())
else:
    print("No record sets were found in the metadata.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records by specific criteria, normalizing a numeric field, and grouping results. All field references should use their exact `@id`s as seen above.

We will demonstrate with the first numeric field found in the RecordSet. Adjust the field `@id` as needed for your analysis.

In [ ]:
# Identify a numeric field (column) in the DataFrame to analyze
import numpy as np

# This cell automatically finds the first float/integer column
numeric_field = None
if len(dataframes) > 0:
    df_cols = dataframes[main_record_set_id].columns
    for col in df_cols:
        # Try to safely cast some values to float
        try:
            s = pd.to_numeric(dataframes[main_record_set_id][col], errors='coerce')
            if np.issubdtype(s.dtype, np.number) and s.notnull().sum() > 0:
                numeric_field = col
                break
        except Exception:
            pass
    if numeric_field:
        print(f"First numeric field selected for EDA: {numeric_field}")
        s = pd.to_numeric(dataframes[main_record_set_id][numeric_field], errors='coerce')

        # Example: Filter for values above the median
        threshold = s.median()
        filtered_df = dataframes[main_record_set_id][s > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (s[s > threshold] - s[s > threshold].mean()) / s[s > threshold].std()
        print(f"Normalized {numeric_field}:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        # Try grouping by the first suitable non-numeric field
        group_field = None
        for col in df_cols:
            if col != numeric_field and dataframes[main_record_set_id][col].dtype == object:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped (mean) {numeric_field} by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable non-numeric field for grouping found.")
    else:
        print("No numeric field found in DataFrame.")
else:
    print("No record sets loaded for EDA.")

## 5. Visualization
Visualize data distributions or field relationships using matplotlib/seaborn. This section uses the selected `numeric_field` (referenced by `@id`) from above. Adjust for your field of interest.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and numeric_field is not None:
    s = pd.to_numeric(dataframes[main_record_set_id][numeric_field], errors='coerce')
    plt.figure(figsize=(8,4))
    sns.histplot(s.dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Plot boxplot grouped by first string field
    if group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=dataframes[main_record_set_id][group_field], y=s)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data loaded for visualization.")

## 6. Conclusion
In this notebook, you explored the FAIR^2 dataset on mass spectrometry of extracellular vesicles using the `mlcroissant` library. You loaded the schema, reviewed data structure via `@id` URIs, extracted records, performed basic data filtering, normalization, grouping, and visualized results. Further domain-specific analysis can build upon these steps by working with field `@id`s for reproducibility in FAIR data science workflows.

*Feel free to rerun cells or adjust the code to focus on other record sets or specific fields of interest (always using their `@id`).*